# Phase 1 — Dataset audit for waste classification

This Colab notebook audits the five classes required by the assignment: **plastic, paper, glass, metal, and organic**. Run it after the Kaggle standardization cell has copied the images into Google Drive.

## Colab preparation

1. In Colab, choose **Runtime → Change runtime type → T4 GPU**. A GPU is not needed for this audit, but it will be ready for later training.
2. Run the Kaggle download-and-standardization cell from the project chat. It copies the five prepared labels into `MyDrive/waste-classification/data/raw/mwcd/`.
3. Check that this folder contains category folders named `plastic`, `paper`, `glass`, `metal`, and `organic`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/waste-classification')
DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'mwcd'
TARGET_CLASSES = ('plastic', 'paper', 'glass', 'metal', 'organic')
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print(f'Dataset folder: {DATASET_ROOT}')
assert DATASET_ROOT.exists(), (
    'Dataset folder not found. Run the Kaggle standardization cell first, or change DATASET_ROOT.'
)

In [ ]:
import pandas as pd
from PIL import Image, UnidentifiedImageError

records, unreadable = [], []
for class_name in TARGET_CLASSES:
    class_dir = DATASET_ROOT / class_name
    if not class_dir.exists():
        raise FileNotFoundError(f'Missing expected class folder: {class_dir}')
    for image_path in class_dir.rglob('*'):
        if image_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        try:
            with Image.open(image_path) as image:
                image.verify()
            records.append({'path': str(image_path), 'label': class_name})
        except (UnidentifiedImageError, OSError, ValueError) as error:
            unreadable.append({'path': str(image_path), 'reason': str(error)})

manifest = pd.DataFrame(records)
class_counts = manifest['label'].value_counts().reindex(TARGET_CLASSES, fill_value=0)
display(class_counts.rename_axis('class').reset_index(name='usable_images'))
print(f'Usable images: {len(manifest):,}')
print(f'Unreadable images: {len(unreadable):,}')
assert manifest['label'].nunique() == 5, 'All five target classes must be present.'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, hue=class_counts.index, legend=False)
plt.title('Usable images per target class')
plt.xlabel('Waste class')
plt.ylabel('Image count')
plt.tight_layout()
plt.show()

In [ ]:
import random

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for axis, class_name in zip(axes, TARGET_CLASSES):
    candidate = random.choice(manifest.loc[manifest['label'] == class_name, 'path'].tolist())
    axis.imshow(Image.open(candidate).convert('RGB'))
    axis.set_title(class_name.title())
    axis.axis('off')
plt.suptitle('One random sample from each target class', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
audit_dir = PROJECT_ROOT / 'outputs' / 'phase1_audit'
audit_dir.mkdir(parents=True, exist_ok=True)
manifest.to_csv(audit_dir / 'image_manifest.csv', index=False)
pd.DataFrame(unreadable).to_csv(audit_dir / 'unreadable_images.csv', index=False)
class_counts.rename('usable_images').to_csv(audit_dir / 'class_counts.csv')
print(f'Audit files saved in: {audit_dir}')